# Intro

## Global parameters

In [1]:
MAX_TIME_HOURS=1
ALPHA=0.01
_PVAL_FLOOR=10**-6

## Modules

### Standard

In [2]:
import os, pickle, platform, sys
import numpy as np

In [3]:
from collections import defaultdict

In [4]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [5]:
import matplotlib.pyplot as plt
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 2
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 2

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['xtick.minor.size'] = 5
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.minor.size'] = 5
plt.rcParams['ytick.minor.width'] = 1
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [6]:
from scipy.stats import spearmanr

In [7]:
from tqdm.notebook import tqdm, trange

In [8]:
import datetime as dt

In [9]:
from bowtie import edges2bowtie

### Home made

In [10]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [11]:
from auxiliary_functions import el2ks

In [12]:
from sam_bowtie import block_and_fluxes as bnf

In [13]:
from plot_bowtie import plot_bowtie_blocks, plot_bowtie_fluxes, _add_colorbar
from plot_bowtie import _fdr as fdr

## Load data

In [14]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
PVALUE_FOLDER=HOME+'pvalues/'
GUARINO_FOLDER=HOME+'guarino_files/'
BIPARTITE_FOLDER=HOME+'BiDCM/'
PLOT_FOLDER=HOME+'plots/'

# Looking into the abyss

In [15]:
guarino_files=[file for file in os.listdir(GUARINO_FOLDER) if not file.startswith('.')]
guarino_files.sort()
guarino_files

['all_dico_labels.txt',
 'crisi_dico_0_bowtie_sizes.csv',
 'crisi_dico_1_bowtie_sizes.csv',
 'crisi_dico_2_bowtie_sizes.csv',
 'crisi_dico_3_bowtie_sizes.csv',
 'crisi_dico_4_bowtie_sizes.csv',
 'crisi_dico_labels.pickle',
 'ita_elections_dico_0_bowtie_sizes.csv',
 'ita_elections_dico_1_bowtie_sizes.csv',
 'ita_elections_dico_2_bowtie_sizes.csv',
 'ita_elections_dico_3_bowtie_sizes.csv',
 'ita_elections_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_5_bowtie_sizes.csv',
 'ita_elections_dico_6_bowtie_sizes.csv',
 'ita_elections_dico_labels.pickle',
 'quirinale_dico_0_bowtie_sizes.csv',
 'quirinale_dico_1_bowtie_sizes.csv',
 'quirinale_dico_2_bowtie_sizes.csv',
 'quirinale_dico_3_bowtie_sizes.csv',
 'quirinale_dico_4_bowtie_sizes.csv',
 'quirinale_dico_5_bowtie_sizes.csv',
 'quirinale_dico_6_bowtie_sizes.csv',
 'quirinale_dico_labels.pickle']

In [16]:
bipartite_files=[file for file in os.listdir(BIPARTITE_FOLDER) if not file.startswith('.')]
bipartite_files.sort()
bipartite_files

['crisi_dico_0_bowtie_flows.csv',
 'crisi_dico_0_bowtie_sizes.csv',
 'crisi_dico_1_bowtie_flows.csv',
 'crisi_dico_1_bowtie_sizes.csv',
 'crisi_dico_2_bowtie_flows.csv',
 'crisi_dico_2_bowtie_sizes.csv',
 'crisi_dico_3_bowtie_flows.csv',
 'crisi_dico_3_bowtie_sizes.csv',
 'crisi_dico_4_bowtie_flows.csv',
 'crisi_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_0_bowtie_flows.csv',
 'ita_elections_dico_0_bowtie_sizes.csv',
 'ita_elections_dico_1_bowtie_flows.csv',
 'ita_elections_dico_1_bowtie_sizes.csv',
 'ita_elections_dico_2_bowtie_flows.csv',
 'ita_elections_dico_2_bowtie_sizes.csv',
 'ita_elections_dico_3_bowtie_flows.csv',
 'ita_elections_dico_3_bowtie_sizes.csv',
 'ita_elections_dico_4_bowtie_flows.csv',
 'ita_elections_dico_4_bowtie_sizes.csv',
 'ita_elections_dico_5_bowtie_flows.csv',
 'ita_elections_dico_5_bowtie_sizes.csv',
 'ita_elections_dico_6_bowtie_flows.csv',
 'ita_elections_dico_6_bowtie_sizes.csv',
 'quirinale_dico_0_bowtie_flows.csv',
 'quirinale_dico_0_bowtie_sizes.cs

### Name of the various dicos

In [17]:
guarino_files[0]

'all_dico_labels.txt'

In [18]:
with open(GUARINO_FOLDER+guarino_files[0], 'r') as f:
    cacca=f.readlines()


In [19]:
def parse_dico_names(filepath):
    with open(filepath, 'r') as f:
        lines = [line.strip().split() for line in f]
        # such a command creates a list
        # in which each element is a list of tokens of a line in the file
        # remarkably, there is an empty element
        # before each dataset name
        

    result = {}
    current_key = None

    for tokens in lines:
        if not tokens:
            current_key = None
        elif current_key is None:
            # prima riga non vuota del gruppo: nome del dataset
            current_key = tokens[0]
            result[current_key] = {}
        else:
            # riga tipo ['5:', 'journalists', '&', 'Media']
            idx = int(tokens[0].rstrip(':'))
            label = ' '.join(tokens[1:])
            result[current_key][idx] = label

    return result

In [22]:
cacca=parse_dico_names(GUARINO_FOLDER+guarino_files[0])

In [23]:
cacca

{'quirinale': {5: 'journalists & Media',
  2: 'M5S',
  0: 'journalists & IV & Azione & +Europa & Media',
  4: 'Lega & FDI',
  1: 'PD',
  3: 'Media & journalists',
  6: 'FI'},
 'crisi': {1: 'Lega & FDI & FI',
  2: 'M5S & journalists',
  0: 'journalists & IV & Media & Azione & +Europa',
  4: 'Media & journalists',
  3: 'PD'},
 'ita_elections': {1: 'PD & Media & +Europa & journalists',
  2: 'M5S & Media',
  3: 'journalists & IV & Azione',
  0: 'Lega & FDI & FI',
  4: 'journalists & Media (1)',
  5: 'Media',
  6: 'journalists & Media (2)'}}

The pickle is what I need.

# Benchmarking: 

In [ ]:
test_files=os.listdir(TEST_FOLDER)
test_files.sort()

## Ita_election
i.e. our main benchmark.

In [ ]:
elected_files=[file for file in test_files if file.startswith('ita')]
elected_files

In [43]:
for file in elected_files:
    with open(TEST_FOLDER+file, 'rb') as f:
        cacca=pickle.load(f)
    file_name_friendly=file[14:25].strip('.')
    if cacca.sol.mre<10**-5:
        print(f"{file_name_friendly}: Ok, MRE={cacca.sol.mre:.1e}")
    else:
        print(f"{file_name_friendly}: Bad! MRE={cacca.sol.mre:.1e} ")

    

dico0_qdecm: Ok, MRE=5.3e-06
dico1_qdecm: Ok, MRE=9.8e-06
dico2_decm: Bad! MRE=8.2e-04 
dico2_qdecm: Ok, MRE=1.1e-06
dico3_decm: Bad! MRE=2.7e-03 
dico3_qdecm: Ok, MRE=4.7e-06
dico4_qdecm: Ok, MRE=9.8e-06
dico5_qdecm: Ok, MRE=3.1e-06
dico6_qdecm: Ok, MRE=4.0e-06


Ok, so far no decm converge with MRE<10^-5. Not good...

## Quirinale

In [53]:
quirinale_files=[file for file in test_files if 'quirinale' in file]

In [54]:
quirinale_files

['quirinale_dico0_qdecm.pkl',
 'quirinale_dico1_qdecm.pkl',
 'quirinale_dico2_qdecm.pkl',
 'quirinale_dico3_qdecm.pkl',
 'quirinale_dico4_qdecm.pkl',
 'quirinale_dico5_qdecm.pkl',
 'quirinale_dico6_qdecm.pkl']

In [56]:
for file in quirinale_files:
    try:
        with open(TEST_FOLDER+file, 'rb') as f:
            cacca=pickle.load(f)
    except Exception as e:
        print(f"Error loading {file}: {e}")
        continue
    file_name_friendly=file[10:21].strip('_')
    if cacca.sol.mre<10**-5:
        print(f"{file_name_friendly}: Ok, MRE={cacca.sol.mre:.1e}")
    else:
        print(f"{file_name_friendly}: Bad! MRE={cacca.sol.mre:.1e} ")

    

dico0_qdecm: Ok, MRE=3.2e-06
dico1_qdecm: Ok, MRE=9.8e-06
dico2_qdecm: Ok, MRE=1.7e-06
dico3_qdecm: Ok, MRE=9.3e-06
dico4_qdecm: Ok, MRE=9.8e-06
dico5_qdecm: Ok, MRE=5.9e-07
dico6_qdecm: Ok, MRE=6.9e-06


## Crisis

In [61]:
crisis_files=[file for file in test_files if 'crisi' in file and file.endswith('.pkl')]

In [62]:
crisis_files

['crisi_dico0_qdecm.pkl',
 'crisi_dico1_qdecm.pkl',
 'crisi_dico2_decm.pkl',
 'crisi_dico2_decm_and_0_gamma_0.0_hub_5.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_0.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_0_0.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_0_gauge_min.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_10.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_20.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_5.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_5_0.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_5_1.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_5_gaugeNone_canonical.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_5_gauge_min.pkl',
 'crisi_dico2_decm_and_10_gamma_1.2_hub_5.pkl',
 'crisi_dico2_decm_and_5_gamma_0.0_hub_5.pkl',
 'crisi_dico2_decm_and_7_gamma_0.0_hub_10.pkl',
 'crisi_dico2_decm_and_7_gamma_0.0_hub_5.pkl',
 'crisi_dico2_qdecm.pkl',
 'crisi_dico3_decm.pkl',
 'crisi_dico3_qdecm.pkl',
 'crisi_dico4_decm.pkl',
 'crisi_dico4_qdecm.pkl',
 'crisis_dwcm_new_gs_nprocs_1.

In [65]:
for file in crisis_files:
    try:
        with open(TEST_FOLDER+file, 'rb') as f:
            cacca=pickle.load(f)
    except Exception as e:
        print(f"Error loading {file}: {e}")
        continue
    file_name_friendly=file[6:17].strip('.').strip('_')
    if cacca.sol.mre<10**-5:
        print(f"{file_name_friendly}: Ok, MRE={cacca.sol.mre:.1e}")
    else:
        print(f"{file_name_friendly}: Bad! MRE={cacca.sol.mre:.1e} ")

    

dico0_qdecm: Ok, MRE=9.7e-06
dico1_qdecm: Ok, MRE=5.9e-06
dico2_decm: Bad! MRE=7.5e-04 
dico2_decm: Bad! MRE=8.7e-02 
dico2_decm: Bad! MRE=5.0e-03 
dico2_decm: Bad! MRE=4.1e-05 
dico2_decm: Bad! MRE=4.2e-05 
dico2_decm: Bad! MRE=1.8e-03 
dico2_decm: Bad! MRE=7.7e-03 
dico2_decm: Ok, MRE=9.3e-06
dico2_decm: Bad! MRE=2.4e-01 
dico2_decm: Bad! MRE=4.1e-05 
dico2_decm: Bad! MRE=4.3e-05 
dico2_decm: Bad! MRE=4.6e-05 
dico2_decm: Bad! MRE=4.8e-03 
dico2_decm: Bad! MRE=1.4e-03 
dico2_decm: Bad! MRE=1.3e-03 
dico2_decm: Bad! MRE=1.1e-03 
dico2_qdecm: Ok, MRE=8.1e-06
dico3_decm: Bad! MRE=2.4e-02 
dico3_qdecm: Ok, MRE=3.5e-06
dico4_decm: Bad! MRE=6.3e-03 
dico4_qdecm: Ok, MRE=9.2e-06
dwcm_new_g: Ok, MRE=2.5e-06
dwcm_new_g: Ok, MRE=2.5e-06
dwcm_new_t: Ok, MRE=1.4e-06
dwcm_new_t: Ok, MRE=1.4e-06
dwcm_old_g: Ok, MRE=3.9e-06
dwcm_old_t: Ok, MRE=2.5e-06
Error loading crisis_qdecm_new_theta_nprocs_0.pkl: No module named 'dcms.models.adecm'
Error loading crisis_qdecm_new_theta_nprocs_0_dico0.pkl: No mo

# Model selection so far
Actually, a proper model selection should be done either on the entire dataset or to each representative of a dataset. So far I have neither the latter, nor the former, therefore, as a rule of thumb, I am using crisi_dico2

In [ ]:
for file in crisis_files:
    if 'dico2' in file:
        try:
            with open(TEST_FOLDER+file, 'rb') as f:
                cacca=pickle.load(f)
        except Exception as e:
            print(f"Error loading {file}: {e}")
            continue
    file_name_friendly=file[6:17].strip('.').strip('_')
    if cacca.sol.mre<10**-5:
        print(f"{file_name_friendly}: Ok, MRE={cacca.sol.mre:.1e}")
    else:
        print(f"{file_name_friendly}: Bad! MRE={cacca.sol.mre:.1e} ")

    